# Atelier : Data Wrangler

**But pédagogique** : utiliser l'interface visuelle **Data Wrangler** dans un notebook Fabric pour explorer, nettoyer et transformer des données, puis récupérer le code Python généré automatiquement.

**Pré-requis** : Lakehouse `dwg_lakehouse` attaché à ce notebook, fichier `consumption_raw.csv` chargé dans **Files** (voir Partie 1).

> Les cellules marquées **🖱️ Data Wrangler** se font dans l'interface (menu **Data → Launch Data Wrangler**). Les cellules de code « générées » correspondent à ce que Data Wrangler produit via **Add code to notebook** : vous pouvez soit les exécuter directement, soit les obtenir vous-même depuis l'interface puis comparer.

## Contexte et données

Le fichier source est **`consumption_raw.csv`** : consommation électrique brute avec des problèmes de qualité volontaires.

| Problème | Exemple |
| --- | --- |
| Formats de dates mixtes | `2025-01-03T04:30:00`, `17/01/2025 18:15:00`, `2025-01-22 01:00:00` |
| Valeur manquante | `consumption_mw` vide (`status = ERROR`) |
| Valeurs aberrantes | `consumption_mw = -888 / -777 / -999` (`status = ERROR`) |
| Doublons | lignes strictement identiques (retransmissions) |
| Négatifs impossibles | `voltage_v` / `frequency_hz` négatifs sur des lignes `OK` |

## Partie 1 — Préparer l'environnement

1. Ouvrir https://app.fabric.microsoft.com puis votre workspace
2. Créer un **Lakehouse** `dwg_lakehouse` (ou utiliser un existant)
3. Dans le Lakehouse, ouvrir **Files** → **Upload** → `consumption_raw.csv`
4. Attacher ce Lakehouse à ce notebook (panneau **Explorer**)

### 1.2 — Charger les données

In [ ]:
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv("Files/consumption_raw.csv")
    .toPandas()
)

display(df)

**À vérifier** : colonnes `timestamp`, `site_id`, `consumption_mw`, `voltage_v`, `frequency_hz`, `status` ; dates hétérogènes ; une ligne avec `consumption_mw` vide et `status = ERROR`.

## Partie 2 — Ouvrir Data Wrangler  🖱️

1. Cliquer sur la cellule contenant `df`
2. Barre d'outils du notebook : **Data** → **Launch Data Wrangler**
3. Choisir le DataFrame **`df`**

Panneau gauche = opérations ; centre = aperçu ; droite = résumé statistique par colonne. Les colonnes avec valeurs manquantes sont signalées.

## Partie 3 — Explorer les données  🖱️

- `timestamp` : type `object` (string), pas encore parsé
- `consumption_mw` : 1 valeur manquante visible, et une valeur aberrante `-888`
- `status` : 2 valeurs distinctes (`OK`, `ERROR`)
- Ces lignes anormales correspondent toutes à `status = ERROR`

Vérification en Python :

In [ ]:
print(df.shape)
print(df['status'].value_counts())
print("consumption_mw vide :", (df['consumption_mw'].isna()).sum())
print("consumption_mw < 0  :", (df['consumption_mw'].astype(float) < 0).sum())

## Partie 4 — Nettoyer avec Data Wrangler  🖱️

**4.1 Supprimer les lignes ERROR** : **Filter** → *Filter rows* → Column `status`, Condition `Not equal to`, Value `ERROR` → **Apply**.

**4.2 Convertir `consumption_mw`** : **Change column type** → `consumption_mw` → `Decimal (float64)` → **Apply**.

**4.3 Standardiser `timestamp`** — ⚠️ *Change column type → DateTime* échoue sur les formats mixtes. Utiliser **Custom operation** :

```python
import pandas as pd
df['timestamp'] = pd.to_datetime(df['timestamp'], format='mixed', dayfirst=True)
```

**4.4 (optionnel) Renommer** `consumption_mw` en `consumption_MW` avec **Rename column**.

## Partie 5 — Code généré (Add code to notebook)

Cliquer **Add code to notebook** : Data Wrangler insère une cellule comme celle-ci.

In [ ]:
# Code generated by Data Wrangler for pandas DataFrame
import pandas as pd

def clean_data(df):
    # Filter rows based on column: 'status'
    df = df[df['status'] != "ERROR"]
    # Change column type to float64 for column: 'consumption_mw'
    df = df.astype({'consumption_mw': 'float64'})
    df['timestamp'] = pd.to_datetime(df['timestamp'], format='mixed', dayfirst=True)
    # Rename column 'consumption_mw' to 'consumption_MW'
    df = df.rename(columns={'consumption_mw': 'consumption_MW'})
    return df

df_clean = clean_data(df.copy())
display(df_clean)

> 💡 Le code est **réutilisable et versionnable** (Git, pipeline).

## Partie 6 — Sauvegarder dans le Lakehouse

In [ ]:
spark_df = spark.createDataFrame(df_clean)

spark_df.write.mode("overwrite").format("delta").saveAsTable("bronze_consumption")

Vérifier dans le Lakehouse que la table **`bronze_consumption`** apparaît sous **Tables**.

---
## Partie 7 — Dédupliquer les lignes (Drop duplicate rows)

**Scénario** : des mesures ont été retransmises ; le fichier contient des lignes strictement identiques que le filtre `ERROR` ne supprime pas.

### 7.1 — Mesurer les doublons

In [ ]:
print("Lignes :", len(df_clean))
print("Doublons exacts :", df_clean.duplicated().sum())

**Attendu** : `Lignes : 17271` et `Doublons exacts : 821`.

### 7.2 — Dédupliquer  🖱️
**Launch Data Wrangler** sur `df_clean` → **Drop duplicate rows** (toutes les colonnes) → **Apply** → **Add code to notebook**. Renommer fonction et sortie pour ne pas écraser `df_clean` :

In [ ]:
def dedup_data(df_clean):
    # Drop duplicate rows across all columns
    df_clean = df_clean.drop_duplicates()
    return df_clean

df_dedup = dedup_data(df_clean.copy())
display(df_dedup)

### 7.3 — Vérifier

Attendu : `16450`, `0`, `0`.

In [ ]:
print("Lignes :", len(df_dedup))
print("Doublons restants :", df_dedup.duplicated().sum())
print("Doublons sur (timestamp, site_id) :", df_dedup.duplicated(subset=["timestamp", "site_id"]).sum())

> 💡 `drop_duplicates(subset=["timestamp", "site_id"], keep="last")` déduplique sur une **clé métier**.

---
## Partie 8 — Remplacer des valeurs aberrantes par des nulls (Find and replace)

**Scénario** : `voltage_v` et `frequency_hz` contiennent des **négatifs impossibles** sur des lignes `OK` :

| Cas | Exemple | Colonnes |
| --- | --- | --- |
| Code d'erreur | `-999.0` | `voltage_v`, `frequency_hz` |
| Erreur de signe | `-228.4` | `voltage_v` |

On remplace la mesure invalide par une valeur nulle (la ligne reste).

### 8.1 — Repérer les négatifs

In [ ]:
print("voltage_v < 0   :", (df_dedup["voltage_v"].astype(float) < 0).sum())
print("frequency_hz < 0:", (df_dedup["frequency_hz"].astype(float) < 0).sum())

**Attendu** : `14` et `5` (dont 8 `-999.0` sur `voltage_v` et 5 sur `frequency_hz`).

### 8.2 — Convertir en numérique  🖱️
**Launch Data Wrangler** sur `df_dedup` → **Change column type** → `voltage_v` puis `frequency_hz` → `Decimal (float64)`. (Le minimum de `voltage_v` apparaît alors à -999.)

### 8.3 — Find and replace  🖱️
**Find and replace** : Target column `voltage_v`, Old value `-999`, New value vide / `NaN` → **Apply** ; idem pour `frequency_hz`.

> ⚠️ Si votre version n'accepte pas une valeur vide comme remplacement, utiliser 8.4.

### 8.4 — Tous les négatifs (Custom operation)  🖱️
Les erreurs de signe ne peuvent pas être listées une à une :

In [ ]:
import numpy as np

def replace_negatives(df_dedup):
    for col in ['voltage_v', 'frequency_hz']:
        s = df_dedup[col].astype('float64')
        df_dedup[col] = s.where(s >= 0, np.nan)
    return df_dedup

df_nonneg = replace_negatives(df_dedup.copy())
print(df_nonneg[["voltage_v", "frequency_hz"]].isna().sum())
print("min voltage_v :", df_nonneg["voltage_v"].min())

**Attendu** : 14 `NaN` dans `voltage_v`, 5 dans `frequency_hz`, minimum de `voltage_v` réaliste (~190 V).

> 💡 `NaN` plutôt que `0` (fausserait les moyennes) ou suppression de ligne (perdrait une consommation valide).

---
## Partie 9 — Découper et fusionner du texte (Split text / Merge)

`site_id` (`SITE_COM_001`) = préfixe + **type** (`COM`, `IND`, `RES`) + **numéro**.

### 9.1 — Split text  🖱️
**Launch Data Wrangler** sur `df_nonneg` → **Split text** (Target column `site_id`, Delimiter `_`) → **Apply** ; renommer les colonnes obtenues en `site_type` et `site_num` (**Rename column**), supprimer la colonne `SITE` (**Drop column**).

### 9.2 — Merge (Custom operation)  🖱️

Code équivalent (les noms de colonnes générés par Split text peuvent varier) :

In [ ]:
def split_merge_site(df_nonneg):
    # Split text on column: 'site_id'
    parts = df_nonneg['site_id'].str.split('_', expand=True)
    df_nonneg['site_type'] = parts[1]
    df_nonneg['site_num'] = parts[2]
    # Merge : libellé lisible
    type_labels = {'COM': 'Commercial', 'IND': 'Industriel', 'RES': 'Residentiel'}
    df_nonneg['site_label'] = df_nonneg['site_type'].map(type_labels) + ' #' + df_nonneg['site_num']
    return df_nonneg

df_sites = split_merge_site(df_nonneg.copy())
display(df_sites[['site_id', 'site_type', 'site_num', 'site_label']].drop_duplicates().sort_values('site_id'))

**Attendu** : 6 lignes, ex. `SITE_COM_001 → COM → 001 → Commercial #001`.

---
## Partie 10 — Colonnes calculées (formule)

`énergie (kWh) = puissance (MW) × 0,25 h × 1000` (mesure toutes les 15 min).

**🖱️** Data Wrangler sur `df_sites` → **Create column from formula** (ou **Custom operation**) : `energy_kwh` = `df['consumption_MW'] * 0.25 * 1000`, puis `date` = `df['timestamp'].dt.date`.

In [ ]:
def add_calculated_columns(df_sites):
    # Energie (kWh) sur un intervalle de 15 min
    df_sites['energy_kwh'] = df_sites['consumption_MW'] * 0.25 * 1000
    df_sites['date'] = df_sites['timestamp'].dt.date
    return df_sites

df_calc = add_calculated_columns(df_sites.copy())
display(df_calc[['timestamp', 'site_id', 'consumption_MW', 'energy_kwh', 'date']].head())

**Vérification** : `0.865` MW → `energy_kwh = 216.25`.

---
## Partie 11 — Agréger avec Group by (DataFrame et fichiers séparés)

**🖱️** Data Wrangler sur `df_calc` → **Group by column and aggregate** : group by `site_id`, `date` ; `consumption_MW` → Mean, Max, Count ; `energy_kwh` → Sum ; `voltage_v` → Mean. Puis **Add code to notebook** et renommer la sortie pour garder `df_calc` intact.

### 11.1 — Agrégat journalier

In [ ]:
def aggregate_daily(df_calc):
    df = df_calc.groupby(['site_id', 'date']).agg(
        consumption_MW_mean=('consumption_MW', 'mean'),
        consumption_MW_max=('consumption_MW', 'max'),
        consumption_MW_count=('consumption_MW', 'count'),
        energy_kwh_sum=('energy_kwh', 'sum'),
        voltage_v_mean=('voltage_v', 'mean'),
    ).reset_index()
    return df

df_daily = aggregate_daily(df_calc.copy())
display(df_daily)
print(df_daily.shape)

**Attendu** : `(180, 7)` — 6 sites × 30 jours ; `df_calc` et `df_daily` coexistent.

### 11.2 — Sauvegarde : fichiers et table séparés

In [ ]:
import os

out_dir = "/lakehouse/default/Files/output"
os.makedirs(out_dir, exist_ok=True)

# Fichiers CSV séparés : détail nettoyé et agrégat journalier
df_calc.to_csv(f"{out_dir}/consumption_clean.csv", index=False)
df_daily.to_csv(f"{out_dir}/consumption_daily_site.csv", index=False)

# Table Delta de l'agrégat
spark.createDataFrame(df_daily).write.mode("overwrite").format("delta").saveAsTable("consumption_daily_site")

**Attendu** : **Files → output** contient `consumption_clean.csv` et `consumption_daily_site.csv` (cliquer **Refresh**) ; la table **`consumption_daily_site`** (180 lignes) apparaît sous **Tables**.

---
## Partie 12 — Copilot dans Data Wrangler  🖱️

**Prérequis** : capacité Fabric **F2+** (ou P) affectée au workspace (pas Trial), paramètre de tenant Copilot activé. Sinon, passer à la Partie 13 (le code manuel ci-dessous donne le même résultat).

**Opération 1** — Data Wrangler sur `df_calc` → **Custom operation** → zone **Copilot**, prompt :

```text
Add a column named period with the value 'night' when the hour of timestamp is between 22 and 5 (inclusive of 22, exclusive of 6), otherwise 'day'
```
**Run Preview** → relire le code → **Apply**.

**Opération 2** — nouvelle **Custom operation**, prompt :

```text
Add a boolean column is_outlier that is True when consumption_MW is greater than the mean plus 3 standard deviations of consumption_MW computed per site_id
```
Vérifier que le calcul est **par site** → **Run Preview** → **Apply** → **Add code to notebook**.

Code de référence (écrit à la main) pour comparer avec celui de Copilot :

In [ ]:
df_copilot = df_calc.copy()

# Opération 1 : période jour/nuit
df_copilot['period'] = df_copilot['timestamp'].dt.hour.apply(lambda h: 'night' if h >= 22 or h < 6 else 'day')

# Opération 2 : valeurs atypiques par site (> moyenne + 3 écarts-types)
g = df_copilot.groupby('site_id')['consumption_MW']
df_copilot['is_outlier'] = df_copilot['consumption_MW'] > (g.transform('mean') + 3 * g.transform('std'))

print(df_copilot['period'].value_counts())
print(df_copilot['is_outlier'].value_counts())

| Question | Réflexe |
| --- | --- |
| Le code fait-il exactement ce que j'ai demandé ? | Relire avant **Apply** |
| Logique par site ou globale ? | Reformuler le prompt si ambigu |
| Résultat identique à l'écriture manuelle ? | Comparer (`value_counts`, `.equals()`) |

> 💡 Prompts courts, une seule action, noms de colonnes explicites ; les prompts en anglais donnent en général de meilleurs résultats.

---
## Partie 13 — Défi autonome

À partir de `df_calc` :
1. Calculer (Data Wrangler → Group by) la consommation **moyenne et maximale par `site_type` et par heure** (extraire `hour` de `timestamp`)
2. Écrire le résultat dans `Files/output/consumption_hourly_type.csv`
3. Ajouter : `assert df_hourly['consumption_MW_mean'].notna().all()`

In [ ]:
# Votre code ici

## Récapitulatif

| Opération | Effet |
| --- | --- |
| Filter rows | Suppression des lignes `ERROR` |
| Change column type → float | `consumption_mw` exploitable |
| Custom operation (to_datetime) | `timestamp` unifié |
| Add code to notebook | Code Python réutilisable |
| Drop duplicate rows | 17 271 → 16 450 lignes |
| Find and replace / Custom op | `-999` et négatifs → `NaN` |
| Split text + merge | `site_type`, `site_num`, `site_label` |
| Formule | `energy_kwh`, `date` |
| Group by | `df_daily` (180 lignes) → CSV + Delta |
| Copilot | `period`, `is_outlier` en langage naturel |